# Solution: Pareto Optimization of SEI Additives with ALCHEMI

Lithium-metal and lithium-ion batteries depend on interfaces. During cycling, electrolyte molecules touch highly reactive electrode surfaces: the negative electrode can drive reduction chemistry, while the positive electrode can drive oxidation chemistry. Some decomposition is harmful because it consumes electrolyte and active lithium, but some early decomposition can be useful when it forms a thin passivating interphase. For battery context, see the SEI modeling review by [Shi et al.](https://www.nature.com/articles/s41524-018-0064-0).

At the anode, this protective layer is usually called the solid electrolyte interphase (SEI). A useful SEI should block electrons, allow Li+ transport, remain chemically and mechanically stable, and avoid continuously attacking the electrolyte; this electronically insulating, ionically conducting passivation picture is reviewed by [Shi et al.](https://www.nature.com/articles/s41524-018-0064-0). Electrolyte additives are often chosen because they react before the bulk solvent and help seed a more protective SEI; additive design and preferential film formation are reviewed by [Balakrishnan et al.](https://www.sciencedirect.com/science/article/pii/S2451910320300089). The design tension is subtle: an additive that barely interacts may do nothing, while one that binds or decomposes too aggressively may create impedance, gas, or unstable products.

A realistic battery-interface simulation would need explicit liquid electrolyte, electron transfer, lithium-ion motion, voltage, many reaction pathways, long timescales, and multiple surface structures. The breadth of these modeling challenges is a central theme in the SEI modeling literature ([Shi et al.](https://www.nature.com/articles/s41524-018-0064-0)). Instead, this notebook demonstrates a small, transparent screening proxy using the skills from Part 1: generate surface+molecule systems, relax them in ALCHEMI Toolkit batches, compute binding energies, and use those energies to rank candidates.

The two challenge objectives encode the tradeoff. First, a molecule should have useful moderate interaction with a reactive Li-metal proxy, which represents SEI seeding. Second, once a passivating SEI-like product exists, the molecule should interact weakly with that surface, which represents compatibility with a protective layer. Because these goals can conflict, this solution uses Pareto hypervolume improvement rather than a single hand-picked threshold; the hypervolume indicator is a standard multi-objective quality measure ([Hypervolume bibliography](https://hypervolume.org/bibliography.html)).

The chemistry here is intentionally simplified. Li metal is a reactive anode proxy. Each molecule class maps to one passivating SEI-product proxy surface using the lookup table in `data/class_surface_lookup.csv`. The bundled structures are a starter panel for a workflow exercise, not production battery-interface reference models. If `data/custom_molecule_manifest.csv` exists, this solution appends those literature/custom candidates before running the workflow; custom rows should record citation/provenance. For example, FEC is included because it is a widely studied SEI additive with reported LiF-containing reduction products ([PNNL summary](https://www.pnnl.gov/publications/reduction-mechanism-fluoroethylene-carbonate-stable-solid-electrolyte-interphase-film)).

The reward functions are deliberately simple but no longer tied to one arbitrary target energy. They use a moderate-adsorption window for Li-metal seeding and a weak-adsorption preference for SEI passivation, following the same qualitative logic used in SEI-additive and Sabatier-style surface-screening literature ([Lee et al.](https://www.frontiersin.org/journals/energy-research/articles/10.3389/fenrg.2021.654460/full)). The exact constants remain a challenge calibration so the task is reproducible and model-free to grade.


## Solution Outputs

This notebook writes `outputs/challenge_submission.csv` and `outputs/raw_component_energies.csv`. The separate grader can check both files without running model calls.


## Control Panel

These settings keep the example small while preserving the Part 1 adsorption-search pattern: relaxed clean slabs, a compact site/orientation grid, batched relaxations, and lowest-energy valid-start selection.


In [ ]:
from pathlib import Path

TOOLKIT_CHECKPOINT = "medium-mpa-0"
TOOLKIT_HEAD = None
TOOLKIT_DEVICE = "auto"
TOOLKIT_DTYPE = "float32"
TOOLKIT_COMPILE_MODEL = False
TOOLKIT_ENABLE_CUEQ = True
TOOLKIT_DT = 0.005
TOOLKIT_N_STEPS = 5000
TOOLKIT_FMAX = 0.05
TOOLKIT_FIRE2_MAXSTEP = 0.04
TOOLKIT_D3BJ = None
BATCH_SIZE = 2

SOLUTION_SETTINGS = {
    "adsorption_height_A": 2.6,
    "li_metal_adsorption_height_A": 2.1,
    "gas_box_A": 20.0,
    "adsorption_site_limit": 3,
    "adsorption_azimuth_angles_deg": (0.0,),
    "max_surface_displacement_A": 1.5,
    "frozen_surface_fraction": 0.5,
}

# Compact smoke-test panel. Add physical slab builders before using the full panel.
EXAMPLE_SYSTEMS = (
    ("FEC", "li_metal"),
    ("FEC", "passivating"),
)

OUTPUT_DIR = Path("outputs")
SUBMISSION_PATH = OUTPUT_DIR / "challenge_submission.csv"
RAW_COMPONENT_ENERGIES_PATH = OUTPUT_DIR / "raw_component_energies.csv"


## Setup

The solution reuses the Part 1 Toolkit backend and keeps challenge-specific geometry/search utilities in `challenge_utils.solution_helpers`.


In [ ]:
import os
import sys
from importlib.metadata import PackageNotFoundError, version

NOTEBOOK_DIR = Path.cwd().resolve()
if not (NOTEBOOK_DIR / "data" / "molecule_manifest.csv").exists():
    candidate = NOTEBOOK_DIR / "challenge-sei"
    if (candidate / "data" / "molecule_manifest.csv").exists():
        NOTEBOOK_DIR = candidate.resolve()
    else:
        raise RuntimeError("Start Jupyter from challenge-sei or from the repository root.")
os.chdir(NOTEBOOK_DIR)
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

os.environ["ALCHEMI_ALLOW_CACHE_OVERWRITE"] = "1"

REPO_ROOT = NOTEBOOK_DIR.parent
PART1_ROOT = REPO_ROOT / "part-1-batched-adsorption"
if not (PART1_ROOT / "helpers" / "__init__.py").exists():
    raise RuntimeError("Cannot find Part 1 helpers. Keep challenge-sei beside part-1-batched-adsorption.")
sys.path.insert(0, str(PART1_ROOT))

import numpy as np
import pandas as pd

from helpers import (
    ToolkitRelaxationConfig,
    ToolkitD3BJConfig,
    check_toolkit_native_api,
    get_toolkit_relaxation_engine,
    display_widgets_grid,
)
from challenge_utils.pareto import dominates, hypervolume_2d, pareto_flags
from challenge_utils.rewards import passivation_score, seeding_score
from challenge_utils.solution_helpers import (
    SolutionSettings,
    binding_energy_table,
    build_adsorption_surfaces,
    choose_inspection_candidates,
    component_energy_table,
    inspection_widget_rows,
    load_atoms,
    make_clean_surface_jobs,
    make_combined_jobs,
    make_gas_jobs,
    prepare_challenge_tables,
    relax_structures,
    require_all_converged,
    select_lowest_energy_site_results,
    selected_site_summary,
    surface_summary,
    write_ovito_inspection_structures,
)

SETTINGS = SolutionSettings(**SOLUTION_SETTINGS)

print(f"Challenge folder : {NOTEBOOK_DIR.name}")
print(f"Part 1 helpers   : {PART1_ROOT.relative_to(REPO_ROOT)}")
for pkg in ("ase", "numpy", "pandas", "torch", "nvalchemi-toolkit", "ovito"):
    try:
        print(f"{pkg:<18}: {version(pkg)}")
    except PackageNotFoundError:
        print(f"{pkg:<18}: not installed")


## 1. Load The Challenge Manifests

The starter panel can be extended with `data/custom_molecule_manifest.csv`. The active run is reduced by `EXAMPLE_SYSTEMS` for quick iteration.


In [ ]:
molecules_df = pd.read_csv("data/molecule_manifest.csv")
custom_manifest_path = Path("data/custom_molecule_manifest.csv")
if custom_manifest_path.exists():
    custom_molecules_df = pd.read_csv(custom_manifest_path)
    molecules_df = pd.concat([molecules_df, custom_molecules_df], ignore_index=True)
    print(f"Loaded {len(custom_molecules_df)} custom/literature molecule row(s).")
else:
    print("No data/custom_molecule_manifest.csv found; using the starter molecule panel.")

if molecules_df["candidate_id"].duplicated().any():
    duplicated = sorted(molecules_df.loc[molecules_df["candidate_id"].duplicated(), "candidate_id"].unique())
    raise RuntimeError(f"Duplicate candidate_id value(s): {duplicated}")

surfaces_df = pd.read_csv("data/surface_manifest.csv")
lookup_df = pd.read_csv("data/class_surface_lookup.csv")
challenge_df, run_systems_df, run_challenge_df = prepare_challenge_tables(
    molecules_df,
    lookup_df,
    EXAMPLE_SYSTEMS,
)

assert set(challenge_df["role"]) == {"baseline", "additive"}
assert {"EC", "EMC"}.issubset(set(challenge_df.loc[challenge_df["role"].eq("baseline"), "candidate_id"]))

print(f"Run systems: {len(run_systems_df)} adsorption system(s); {len(run_challenge_df)} molecule reference(s).")
display(challenge_df[["candidate_id", "role", "molecule_class", "passivating_surface_id", "structure_path"]])
display(surfaces_df[["surface_id", "role", "structure_path"]])
display(run_systems_df[["candidate_id", "interaction", "surface_id", "role", "structure_path"]])


## 2. Build The Toolkit Relaxation Engine

This is the same native Toolkit path used in Part 1.


In [ ]:
status = check_toolkit_native_api()
print(status["message"])
if not status["available"]:
    raise RuntimeError("ALCHEMI Toolkit native API is not available in this kernel.")

if isinstance(TOOLKIT_D3BJ, dict):
    TOOLKIT_D3BJ = ToolkitD3BJConfig(**TOOLKIT_D3BJ)

relaxation_config = ToolkitRelaxationConfig(
    name="toolkit",
    cache_dir=(OUTPUT_DIR / "cache_json").as_posix(),
    use_cached_responses=False,
    toolkit_checkpoint=TOOLKIT_CHECKPOINT,
    toolkit_head=TOOLKIT_HEAD,
    toolkit_device=TOOLKIT_DEVICE,
    toolkit_dtype=TOOLKIT_DTYPE,
    toolkit_compile_model=TOOLKIT_COMPILE_MODEL,
    toolkit_enable_cueq=TOOLKIT_ENABLE_CUEQ,
    toolkit_dt=TOOLKIT_DT,
    toolkit_n_steps=TOOLKIT_N_STEPS,
    toolkit_fmax=TOOLKIT_FMAX,
    toolkit_fire2_maxstep=TOOLKIT_FIRE2_MAXSTEP,
    toolkit_d3bj=TOOLKIT_D3BJ,
    toolkit_require_d3bj=TOOLKIT_D3BJ is not None,
)
RELAXATION_ENGINE = get_toolkit_relaxation_engine(relaxation_config)
print(f"Toolkit relaxation engine ready: {RELAXATION_ENGINE.name}")


## 3. Build And Relax The Jobs

As in Part 1, clean slabs are relaxed first. Adsorption starts are then built on the relaxed slabs as a compact `site x orientation x rotation x height` grid.


In [ ]:
molecule_atoms = {
    row.candidate_id: load_atoms(row.structure_path)
    for row in run_challenge_df.itertuples(index=False)
}
surface_meta = surfaces_df.set_index("surface_id")
used_surface_ids = sorted(run_systems_df["surface_id"].unique())

adsorption_surface_atoms = build_adsorption_surfaces(used_surface_ids, settings=SETTINGS)
display(surface_summary(adsorption_surface_atoms, settings=SETTINGS))

gas_jobs = make_gas_jobs(run_challenge_df, molecule_atoms, settings=SETTINGS)
clean_surface_jobs = make_clean_surface_jobs(
    used_surface_ids,
    adsorption_surface_atoms,
    surface_meta,
    settings=SETTINGS,
)
print(f"Gas jobs           : {len(gas_jobs)}")
print(f"Clean-surface jobs : {len(clean_surface_jobs)}")

OUTPUT_DIR.mkdir(exist_ok=True)
gas_results = relax_structures(
    gas_jobs,
    RELAXATION_ENGINE,
    settings=SETTINGS,
    batch_size=BATCH_SIZE,
    label_prefix="solution_sei_gas",
)
clean_surface_results = relax_structures(
    clean_surface_jobs,
    RELAXATION_ENGINE,
    settings=SETTINGS,
    batch_size=BATCH_SIZE,
    label_prefix="solution_sei_clean_surface",
)
require_all_converged(gas_results, label="gas references")
require_all_converged(clean_surface_results, label="clean-surface references")

combined_jobs = make_combined_jobs(
    run_systems_df,
    molecule_atoms,
    surface_meta,
    clean_surface_results,
    settings=SETTINGS,
)
print(f"Combined starts    : {len(combined_jobs)}")

all_combined_results = relax_structures(
    combined_jobs,
    RELAXATION_ENGINE,
    settings=SETTINGS,
    batch_size=BATCH_SIZE,
    label_prefix="solution_sei_combined",
)
combined_results = select_lowest_energy_site_results(all_combined_results)
require_all_converged(combined_results, label="selected combined adsorption systems")
display(selected_site_summary(combined_results))
print("Relaxations complete and converged.")


## 4. Compute Binding Energies

Use the same adsorption-energy convention as Part 1: `E_bind = E_surface+species - E_surface - E_species`.


In [ ]:
raw_component_energies_df = component_energy_table(
    run_systems_df,
    gas_results,
    clean_surface_results,
    combined_results,
)
raw_component_energies_df.to_csv(RAW_COMPONENT_ENERGIES_PATH, index=False)
print(f"Wrote {RAW_COMPONENT_ENERGIES_PATH}")
display(raw_component_energies_df)

binding_df = binding_energy_table(run_challenge_df, raw_component_energies_df)
display(binding_df[["candidate_id", "role", "E_bind_Li_eV", "E_bind_passivating_eV"]])


## 5. Inspect Relaxed Geometries

Write the selected relaxed adsorption structures as EXTXYZ and show them with the Part 1 OVITO widget helper when available.


In [ ]:
OVITO_STRUCTURE_DIR = OUTPUT_DIR / "ovito_structures"
inspection_df = write_ovito_inspection_structures(combined_results, output_dir=OVITO_STRUCTURE_DIR)
INSPECT_CANDIDATE_IDS = choose_inspection_candidates(binding_df)

print(f"Wrote {len(inspection_df)} OVITO structure file(s) to {OVITO_STRUCTURE_DIR}")
print("Inspecting:", ", ".join(INSPECT_CANDIDATE_IDS))
display(inspection_df[inspection_df["candidate_id"].isin(INSPECT_CANDIDATE_IDS)])

try:
    display_widgets_grid(
        inspection_widget_rows(inspection_df, INSPECT_CANDIDATE_IDS),
        width="390px",
        height="310px",
        show_cell=True,
    )
except Exception as exc:
    print(f"OVITO widget display unavailable: {type(exc).__name__}: {exc}")
    display(inspection_df[["candidate_id", "interaction", "surface_id", "structure_path"]])


## 6. Compute Reward Scores

The challenge rubric rewards moderate Li-metal binding for SEI seeding and weak binding on the passivating proxy surface.


In [ ]:
scored_df = binding_df.copy()
scored_df["seeding_score"] = scored_df["E_bind_Li_eV"].map(seeding_score)
scored_df["passivation_score"] = scored_df["E_bind_passivating_eV"].map(passivation_score)

display(scored_df[["candidate_id", "role", "seeding_score", "passivation_score"]])


## 7. Pareto Front And Hypervolume

Treat both scores as objectives to maximize. Hypervolume improvement is measured against the baseline `EC`/`EMC` front with reference point `(0, 0)`.


In [ ]:
final_df = scored_df.copy()
points = list(zip(final_df["seeding_score"], final_df["passivation_score"]))
final_df["is_pareto"] = pareto_flags(points)

baseline_points = list(zip(
    final_df.loc[final_df["role"].eq("baseline"), "seeding_score"],
    final_df.loc[final_df["role"].eq("baseline"), "passivation_score"],
))
baseline_hv = hypervolume_2d(baseline_points)

final_df["hypervolume_improvement"] = [
    0.0 if row.role == "baseline"
    else hypervolume_2d([*baseline_points, (row.seeding_score, row.passivation_score)]) - baseline_hv
    for row in final_df.itertuples(index=False)
]

print(f"Baseline hypervolume: {baseline_hv:.4f}")
display(final_df.sort_values("hypervolume_improvement", ascending=False))


## 8. Select Your Additive And Submit

Mark exactly one additive as selected: the additive with the largest hypervolume improvement.


In [ ]:
submission = final_df.copy()
additives = submission[submission["role"].eq("additive")].copy()
if additives.empty:
    raise RuntimeError("No additive rows are available to select.")

selected_id = (
    additives
    .sort_values(["hypervolume_improvement", "candidate_id"], ascending=[False, True])
    .iloc[0]["candidate_id"]
)
submission["selected"] = submission["candidate_id"].eq(selected_id)

print(f"Selected additive: {selected_id}")
display(submission[[
    "candidate_id", "role", "seeding_score", "passivation_score",
    "is_pareto", "hypervolume_improvement", "selected",
]].sort_values("hypervolume_improvement", ascending=False))


In [ ]:
required_columns = [
    "candidate_id", "role", "molecule_class", "passivating_surface_id",
    "E_bind_Li_eV", "E_bind_passivating_eV", "seeding_score",
    "passivation_score", "is_pareto", "hypervolume_improvement", "selected",
]
missing = [column for column in required_columns if column not in submission.columns]
if missing:
    raise RuntimeError(f"Submission is missing required columns: {missing}")
if int(submission["selected"].sum()) != 1:
    raise RuntimeError("Exactly one row must be selected.")

OUTPUT_DIR.mkdir(exist_ok=True)
submission[required_columns].to_csv(SUBMISSION_PATH, index=False)
print(f"Wrote {SUBMISSION_PATH}")
display(submission[required_columns])


## References And Further Reading

- NVIDIA [ALCHEMI Toolkit documentation](https://nvidia.github.io/nvalchemi-toolkit/) for `AtomicData`, `Batch`, model wrappers, and Toolkit dynamics.
- Batatia et al., [MACE: Higher Order Equivariant Message Passing Neural Networks for Fast and Accurate Force Fields](https://openreview.net/forum?id=YPpSngE-ZU), NeurIPS 2022.
- Larsen et al., [The Atomic Simulation Environment - a Python library for working with atoms](https://doi.org/10.1088/1361-648X/aa680e), J. Phys.: Condens. Matter 2017.
- Stukowski, [Visualization and analysis of atomistic simulation data with OVITO - the Open Visualization Tool](https://doi.org/10.1088/0965-0393/18/1/015012), Modelling Simul. Mater. Sci. Eng. 2010.
- Leung et al., [Stability of Solid Electrolyte Interphase Components on Lithium Metal and Reactive Anode Material Surfaces](https://doi.org/10.1021/acs.jpcc.5b11719), J. Phys. Chem. C 2016; examples use large periodic SEI/Li cells and matching slab-interface references.
- Chanussot et al., [The Open Catalyst 2020 Dataset and Community Challenges](https://doi.org/10.1021/acscatal.0c04525), ACS Catalysis 2021; summarizes common slab-adsorbate setup, adsorption-energy references, vacuum, and fixed subsurface atoms.
- Shi et al., [Review on modeling of the anode solid electrolyte interphase (SEI) for lithium-ion batteries](https://www.nature.com/articles/s41524-018-0064-0), npj Computational Materials 2018.
- Xu et al., [A review on electrolyte additives for lithium-ion batteries](https://www.sciencedirect.com/science/article/pii/S0378775306017538), J. Power Sources 2007.
- Balakrishnan et al., [Electrolyte additives for improved lithium-ion battery performance and overcharge protection](https://www.sciencedirect.com/science/article/pii/S2451910320300089), 2020.
- Li et al., [A Review of Solid Electrolyte Interphases on Lithium Metal Anode](https://pmc.ncbi.nlm.nih.gov/articles/PMC5063117/), Advanced Science 2016.
- [Insights into the efficient roles of solid electrolyte interphase derived from vinylene carbonate additive in rechargeable batteries](https://www.sciencedirect.com/science/article/abs/pii/S1572665722001187), 2022.
- Zhang et al., [Reduction Mechanism of Fluoroethylene Carbonate for Stable Solid-Electrolyte Interphase Film on Silicon Anode](https://www.pnnl.gov/publications/reduction-mechanism-fluoroethylene-carbonate-stable-solid-electrolyte-interphase-film), ChemSusChem 2013.
- Lee et al., [The Sabatier Principle in Electrocatalysis: Basics, Limitations, and Extensions](https://www.frontiersin.org/journals/energy-research/articles/10.3389/fenrg.2021.654460/full), Frontiers in Energy Research 2021.
- Aich et al., [Determination of thermodynamic parameters in adsorption studies: a review](https://link.springer.com/article/10.1007/s11696-025-04218-x), Chemical Papers 2025.
- [Hypervolume bibliography](https://hypervolume.org/bibliography.html) for Pareto hypervolume indicator references.
